<a href="https://colab.research.google.com/github/weloo11/mnist-image-classifier/blob/main/Final%20Sameh%20phase%201.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import numpy as np
import pandas as pd
from tensorflow.keras.datasets import mnist
from sklearn.decomposition import PCA
from skimage.feature import hog




def flatten_features(images):
    return images.reshape(images.shape[0], -1)


def hog_single_image(image):
    return hog(
        image,
        orientations=9,
        pixels_per_cell=(4, 4),
        cells_per_block=(2, 2),
        block_norm="L2-Hys"
    )


def hog_features_dataset(images):
    return np.array([hog_single_image(img) for img in images], dtype=np.float64)



def stratified_train_val_split(x_train, y_train, val_ratio=0.15, random_state=42):
    np.random.seed(random_state)

    class_0_idx = np.where(y_train == 0)[0]
    class_1_idx = np.where(y_train == 1)[0]

    np.random.shuffle(class_0_idx)
    np.random.shuffle(class_1_idx)

    val_0_size = int(len(class_0_idx) * val_ratio)
    val_1_size = int(len(class_1_idx) * val_ratio)

    val_idx = np.concatenate([
        class_0_idx[:val_0_size],
        class_1_idx[:val_1_size]
    ])

    train_idx = np.concatenate([
        class_0_idx[val_0_size:],
        class_1_idx[val_1_size:]
    ])

    np.random.shuffle(train_idx)
    np.random.shuffle(val_idx)

    return (
        x_train[train_idx],
        y_train[train_idx],
        x_train[val_idx],
        y_train[val_idx]
    )




def preprocess_mnist(
    target_digit=5,
    method="hog",
    pca_components=100,
    val_ratio=0.15
):
    (x_train, y_train), (x_test, y_test) = mnist.load_data()

    y_train = np.where(y_train == target_digit, 1, 0)
    y_test = np.where(y_test == target_digit, 1, 0)

    x_train = x_train.astype(np.float64) / 255.0
    x_test = x_test.astype(np.float64) / 255.0

    x_train, y_train, x_val, y_val = stratified_train_val_split(
        x_train,
        y_train,
        val_ratio=val_ratio,
        random_state=42
    )

    if method == "flatten":
        print("Using Flatten features")

        X_train = flatten_features(x_train)
        X_val = flatten_features(x_val)
        X_test = flatten_features(x_test)

        pca_used = "—"

    elif method == "pca":
        print("Using PCA features")

        X_train_flat = flatten_features(x_train)
        X_val_flat = flatten_features(x_val)
        X_test_flat = flatten_features(x_test)

        pca = PCA(n_components=pca_components)

        X_train = pca.fit_transform(X_train_flat)
        X_val = pca.transform(X_val_flat)
        X_test = pca.transform(X_test_flat)

        pca_used = pca_components

        print("PCA variance kept:", round(np.sum(pca.explained_variance_ratio_), 4))

    elif method == "hog":
        print("Using improved HOG features")

        X_train = hog_features_dataset(x_train)
        X_val = hog_features_dataset(x_val)
        X_test = hog_features_dataset(x_test)

        pca_used = "—"

    else:
        raise ValueError("method must be 'flatten', 'pca', or 'hog'")

    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)
    std[std == 0] = 1.0

    X_train = (X_train - mean) / std
    X_val = (X_val - mean) / std
    X_test = (X_test - mean) / std

    print("X_train:", X_train.shape)
    print("X_val:", X_val.shape)
    print("X_test:", X_test.shape)

    print("Train positives:", np.sum(y_train == 1))
    print("Train negatives:", np.sum(y_train == 0))
    print("Val positives:", np.sum(y_val == 1))
    print("Val negatives:", np.sum(y_val == 0))
    print("Test positives:", np.sum(y_test == 1))
    print("Test negatives:", np.sum(y_test == 0))

    return X_train, y_train, X_val, y_val, X_test, y_test, mean, std, pca_used




def confusion_matrix_manual(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    return np.array([[tn, fp],
                     [fn, tp]])


def accuracy_score_manual(y_true, y_pred):
    return np.mean(y_true == y_pred)


def precision_score_manual(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))

    if tp + fp == 0:
        return 0.0

    return tp / (tp + fp)


def recall_score_manual(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    if tp + fn == 0:
        return 0.0

    return tp / (tp + fn)


def f1_score_manual(y_true, y_pred):
    precision = precision_score_manual(y_true, y_pred)
    recall = recall_score_manual(y_true, y_pred)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)




class LinearSVMFromScratch:
    def __init__(
        self,
        learning_rate=0.01,
        lambda_param=0.0005,
        n_epochs=60,
        batch_size=512,
        lr_decay=0.95,
        patience=8,
        pos_weight_scale=1.0
    ):
        self.learning_rate = learning_rate
        self.lambda_param = lambda_param
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.lr_decay = lr_decay
        self.patience = patience
        self.pos_weight_scale = pos_weight_scale

        self.w = None
        self.b = 0.0

        self.train_losses = []
        self.best_w = None
        self.best_b = None
        self.best_f1 = -1.0
        self.best_epoch = 0

    def _compute_class_weights(self, y):
        n_samples = len(y)
        n_pos = np.sum(y == 1)
        n_neg = np.sum(y == -1)

        weight_pos = n_samples / (2 * n_pos)
        weight_neg = n_samples / (2 * n_neg)

        weight_pos *= self.pos_weight_scale

        return weight_pos, weight_neg

    def _hinge_loss(self, X, y, weight_pos, weight_neg):
        scores = X @ self.w + self.b
        margins = y * scores

        sample_weights = np.where(y == 1, weight_pos, weight_neg)
        hinge = np.maximum(0, 1 - margins)

        loss = self.lambda_param * np.dot(self.w, self.w) + np.mean(sample_weights * hinge)

        return loss

    def fit(self, X, y, X_val=None, y_val=None):
        n_samples, n_features = X.shape

        self.w = np.zeros(n_features, dtype=np.float64)
        self.b = 0.0

        weight_pos, weight_neg = self._compute_class_weights(y)

        current_lr = self.learning_rate
        no_improve_count = 0

        for epoch in range(self.n_epochs):
            indices = np.random.permutation(n_samples)

            X_shuffled = X[indices]
            y_shuffled = y[indices]

            for start in range(0, n_samples, self.batch_size):
                end = start + self.batch_size

                X_batch = X_shuffled[start:end]
                y_batch = y_shuffled[start:end]

                scores = X_batch @ self.w + self.b
                margins = y_batch * scores

                sample_weights = np.where(y_batch == 1, weight_pos, weight_neg)
                active = margins < 1

                if np.any(active):
                    X_active = X_batch[active]
                    y_active = y_batch[active]
                    w_active = sample_weights[active]

                    grad_w_hinge = -np.mean(
                        (w_active * y_active)[:, np.newaxis] * X_active,
                        axis=0
                    )

                    grad_b_hinge = -np.mean(w_active * y_active)

                else:
                    grad_w_hinge = np.zeros_like(self.w)
                    grad_b_hinge = 0.0

                grad_w = 2 * self.lambda_param * self.w + grad_w_hinge
                grad_b = grad_b_hinge

                self.w -= current_lr * grad_w
                self.b -= current_lr * grad_b

            train_loss = self._hinge_loss(X, y, weight_pos, weight_neg)
            self.train_losses.append(train_loss)

            msg = f"Epoch {epoch + 1}/{self.n_epochs}, Loss: {train_loss:.6f}"

            if X_val is not None and y_val is not None:
                y_val_pred = self.predict(X_val)

                y_val_binary = np.where(y_val == 1, 1, 0)
                y_val_pred_binary = np.where(y_val_pred == 1, 1, 0)

                val_f1 = f1_score_manual(y_val_binary, y_val_pred_binary)

                msg += f", Val F1: {val_f1:.6f}"

                if val_f1 > self.best_f1:
                    self.best_f1 = val_f1
                    self.best_w = self.w.copy()
                    self.best_b = self.b
                    self.best_epoch = epoch + 1
                    no_improve_count = 0
                else:
                    no_improve_count += 1

                if no_improve_count >= self.patience:
                    print(msg)
                    print("Early stopping triggered.")
                    break

            print(msg)

            current_lr *= self.lr_decay

        if self.best_w is not None:
            self.w = self.best_w
            self.b = self.best_b

    def decision_function(self, X):
        return X @ self.w + self.b

    def predict(self, X):
        scores = self.decision_function(X)
        return np.where(scores >= 0, 1, -1)




method = "hog"
target_digit = 5

X_train, y_train, X_val, y_val, X_test, y_test, mean, std, pca_used = preprocess_mnist(
    target_digit=target_digit,
    method=method,
    pca_components=100,
    val_ratio=0.15
)

y_train_svm = np.where(y_train == 1, 1, -1)
y_val_svm = np.where(y_val == 1, 1, -1)
y_test_svm = np.where(y_test == 1, 1, -1)





learning_rates = [0.005, 0.01, 0.02]
lambda_values = [0.0001, 0.0005, 0.001]
pos_weight_scales = [0.75, 1.0, 1.25]

best_model = None
best_config = None
best_val_f1 = -1.0

results = []

for lr in learning_rates:
    for lam in lambda_values:
        for pos_scale in pos_weight_scales:

            print("\n=================================================")
            print("Testing:")
            print("learning_rate    =", lr)
            print("lambda_param     =", lam)
            print("pos_weight_scale =", pos_scale)
            print("=================================================")

            model = LinearSVMFromScratch(
                learning_rate=lr,
                lambda_param=lam,
                n_epochs=60,
                batch_size=512,
                lr_decay=0.95,
                patience=8,
                pos_weight_scale=pos_scale
            )

            model.fit(
                X_train,
                y_train_svm,
                X_val=X_val,
                y_val=y_val_svm
            )

            y_val_pred_svm = model.predict(X_val)
            y_val_pred = np.where(y_val_pred_svm == 1, 1, 0)

            val_acc = accuracy_score_manual(y_val, y_val_pred)
            val_prec = precision_score_manual(y_val, y_val_pred)
            val_rec = recall_score_manual(y_val, y_val_pred)
            val_f1 = f1_score_manual(y_val, y_val_pred)

            results.append({
                "Learning Rate": lr,
                "Lambda": lam,
                "Positive Weight Scale": pos_scale,
                "Validation Accuracy": val_acc,
                "Validation Precision": val_prec,
                "Validation Recall": val_rec,
                "Validation F1": val_f1,
                "Best Epoch": model.best_epoch
            })

            print("Validation F1:", val_f1)

            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_model = model
                best_config = {
                    "learning_rate": lr,
                    "lambda_param": lam,
                    "pos_weight_scale": pos_scale,
                    "best_epoch": model.best_epoch
                }






svm_model = best_model

print("\n\n================ AUTO-TUNING RESULTS ================")
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

print("\nBest Config:")
print(best_config)
print("Best Validation F1:", best_val_f1)





y_val_pred_svm = svm_model.predict(X_val)
y_val_pred = np.where(y_val_pred_svm == 1, 1, 0)

val_acc = accuracy_score_manual(y_val, y_val_pred)
val_prec = precision_score_manual(y_val, y_val_pred)
val_rec = recall_score_manual(y_val, y_val_pred)
val_f1 = f1_score_manual(y_val, y_val_pred)
val_cm = confusion_matrix_manual(y_val, y_val_pred)

print("\n===== BEST VALIDATION RESULTS =====")
print("Accuracy :", val_acc)
print("Precision:", val_prec)
print("Recall   :", val_rec)
print("F1-score :", val_f1)
print("Confusion Matrix:\n", val_cm)






y_test_pred_svm = svm_model.predict(X_test)
y_test_pred = np.where(y_test_pred_svm == 1, 1, 0)

test_acc = accuracy_score_manual(y_test, y_test_pred)
test_prec = precision_score_manual(y_test, y_test_pred)
test_rec = recall_score_manual(y_test, y_test_pred)
test_f1 = f1_score_manual(y_test, y_test_pred)
test_cm = confusion_matrix_manual(y_test, y_test_pred)

print("\n===== FINAL TEST RESULTS =====")
print("Accuracy :", test_acc)
print("Precision:", test_prec)
print("Recall   :", test_rec)
print("F1-score :", test_f1)
print("Confusion Matrix:\n", test_cm)






summary_table = pd.DataFrame([
    {
        "Feature Method": method.upper(),
        "Best Learning Rate": best_config["learning_rate"],
        "Best PCA Components": pca_used,
        "Lambda": best_config["lambda_param"],
        "Positive Weight Scale": best_config["pos_weight_scale"],
        "Accuracy": round(test_acc, 4),
        "Precision": round(test_prec, 4),
        "Recall": round(test_rec, 4),
        "F1-score": round(test_f1, 4),
        "Best Epoch": best_config["best_epoch"]
    }
])

print("\n\nTable 1 — Linear SVM Final Results")
print(summary_table.to_string(index=False))


tuning_table = pd.DataFrame([
    {
        "Stage": "Feature extraction",
        "Best choice": method.upper(),
        "Result": "Selected feature representation"
    },
    {
        "Stage": "Learning rate tuning",
        "Best choice": best_config["learning_rate"],
        "Result": "Best validation F1-score"
    },
    {
        "Stage": "Regularization tuning",
        "Best choice": best_config["lambda_param"],
        "Result": "Best lambda parameter"
    },
    {
        "Stage": "Class imbalance tuning",
        "Best choice": best_config["pos_weight_scale"],
        "Result": "Best positive class weight scale"
    }
])

print("\n\nTable 2 — Linear SVM Tuning Summary")
print(tuning_table.to_string(index=False))


print("\n\nFinal Test Confusion Matrix")
print(test_cm)

Using improved HOG features
X_train: (51001, 1296)
X_val: (8999, 1296)
X_test: (10000, 1296)
Train positives: 4608
Train negatives: 46393
Val positives: 813
Val negatives: 8186
Test positives: 892
Test negatives: 9108

Testing:
learning_rate    = 0.005
lambda_param     = 0.0001
pos_weight_scale = 0.75
Epoch 1/60, Loss: 0.239107, Val F1: 0.554340
Epoch 2/60, Loss: 0.149821, Val F1: 0.734212
Epoch 3/60, Loss: 0.102281, Val F1: 0.832124
Epoch 4/60, Loss: 0.074331, Val F1: 0.878075
Epoch 5/60, Loss: 0.058413, Val F1: 0.905192
Epoch 6/60, Loss: 0.049528, Val F1: 0.922369
Epoch 7/60, Loss: 0.043037, Val F1: 0.927991
Epoch 8/60, Loss: 0.040167, Val F1: 0.921219
Epoch 9/60, Loss: 0.036012, Val F1: 0.941246
Epoch 10/60, Loss: 0.032743, Val F1: 0.945755
Epoch 11/60, Loss: 0.031073, Val F1: 0.942285
Epoch 12/60, Loss: 0.029231, Val F1: 0.946809
Epoch 13/60, Loss: 0.027429, Val F1: 0.948490
Epoch 14/60, Loss: 0.027119, Val F1: 0.935352
Epoch 15/60, Loss: 0.025921, Val F1: 0.941730
Epoch 16/60, Los